# 03_city_trend_analysis

## 目的
- 特定市区町村や point_id の時系列分析
- 年比較 (compare_years)
- 近傍地点比較
- 公示価格トレンドの探索分析

## 前提
複数年分のデータが DB にあること。まず 01_sync を複数年で実行してください。

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# プロジェクトルートを config.py の存在で探索（CWD に依存しない）
def _find_project_root() -> str:
    # Jupyter が提供する _dh（notebook ディレクトリ）を優先
    try:
        _nb_dir = Path(globals()["_dh"][0])
    except (KeyError, IndexError):
        _nb_dir = Path.cwd()
    for candidate in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]:
        if (candidate / "config.py").exists():
            return str(candidate.resolve())
    # フォールバック: 絶対パスで直接指定
    fallback = Path("/home/kazumasa/projects/land_price_api_app")
    if fallback.exists():
        return str(fallback)
    raise FileNotFoundError("プロジェクトルートが見つかりません")

_project_root = _find_project_root()
os.chdir(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)
print("作業ディレクトリ:", os.getcwd())

作業ディレクトリ: /home/kazumasa/projects/land_price_api_app


## 1. DB 接続・全年度データ読み込み

In [2]:
from notebook_utils import load_env_and_connect
import db
import pandas as pd
from pathlib import Path
from config import PROCESSED_DIR

conn = load_env_and_connect(read_only=True)

if conn is not None:
    years = db.get_available_years(conn)
else:
    pqs = sorted(PROCESSED_DIR.glob("land_prices_y*_pc0.parquet"))
    years = sorted(set(int(p.stem.split("_y")[1].split("_")[0]) for p in pqs), reverse=True)

print("格納済み年度:", years)

if not years:
    print("データがありません。先に同期を実行してください。")
else:
    # 全年度を Parquet または DB から読み込む
    if conn is not None:
        df_all = db.read_land_prices(conn)
    else:
        dfs = [pd.read_parquet(p) for p in sorted(PROCESSED_DIR.glob("land_prices_y*_pc0.parquet"))]
        df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"全データ: {len(df_all):,} 件")

✓ APIキーを確認しました
✓ DB 接続成功 (読み取り専用): 8,188 レコード / 年度: [2026, 2025]
格納済み年度: [2026, 2025]
全データ: 8,188 件


## 2. 市区町村時系列（例: 那覇市）

In [3]:
from analytics import compute_city_summary
import plotly.express as px

if "df_all" in dir() and not df_all.empty:
    CITY = "那覇市"  # ← 変更可能 (例: "千代田区", "那覇市", "石垣市")
    city_df = compute_city_summary(
        df_all[df_all["city_name"].str.contains(CITY, na=False)]
    )
    if city_df.empty:
        print(f"{CITY} のデータがありません")
        print("沖縄データが未取得の場合は 01_sync_and_normalize.ipynb で PREF_CODE='47' を実行してください")
    else:
        fig = px.line(
            city_df.sort_values("year"),
            x="year", y="avg_price",
            color="use_category_name",
            title=f"{CITY} 平均公示価格 推移",
            markers=True,
        )
        fig.update_layout(yaxis_tickformat=",.0f")
        fig.show()

那覇市 のデータがありません
沖縄データが未取得の場合は 01_sync_and_normalize.ipynb で PREF_CODE='47' を実行してください


## 3. 特定地点 (point_id) の年次推移

In [4]:
from notebook_utils import plot_point_history

if "df_all" in dir() and not df_all.empty:
    # point_id の例 (DB の実際の値に置き換えてください)
    sample_pid = df_all["point_id"].dropna().iloc[0] if len(df_all) > 0 else None
    if sample_pid:
        print("表示する point_id:", sample_pid)
        plot_point_history(df_all, sample_pid)
    else:
        print("point_id が取得できません")

表示する point_id: 10005110


## 4. 年比較（2025 vs 2026）

In [5]:
from analytics import compare_years

if "df_all" in dir() and len(years) >= 2:
    year_a, year_b = sorted(years)[-2], sorted(years)[-1]
    comp = compare_years(df_all, year_a, year_b)
    print(f"{year_a} vs {year_b}: {len(comp):,} 地点")
    print("\n上昇TOP10:")
    display(comp.head(10)[["location_text", "city_name", f"price_{year_a}", f"price_{year_b}", "pct_change"]])
    print("\n下落TOP10:")
    display(comp.tail(10)[["location_text", "city_name", f"price_{year_a}", f"price_{year_b}", "pct_change"]])
else:
    print("年比較には 2 年分以上のデータが必要です")

2025 vs 2026: 4,073 地点

上昇TOP10:


,location_text,city_name,price_2025,price_2026,pct_change
0,東京都渋谷区桜丘町１５番６外,渋谷区,3450000.0,4450000.0,28.99
1,東京都台東区浅草１丁目１６番１４外,台東区,7170000.0,9150000.0,27.62
2,東京都台東区西浅草２丁目６６番２,台東区,2580000.0,3230000.0,25.19
3,東京都台東区浅草２丁目２４番６,台東区,998000.0,1240000.0,24.25
4,東京都渋谷区渋谷２丁目６番９外,渋谷区,2900000.0,3600000.0,24.14
5,東京都中野区中野３丁目１０７番１０外,中野区,6670000.0,8270000.0,23.99
6,東京都台東区浅草２丁目５２番１１,台東区,2010000.0,2490000.0,23.88
7,東京都台東区西浅草３丁目２番１０,台東区,1300000.0,1610000.0,23.85
8,東京都台東区浅草５丁目７７番１５,台東区,798000.0,988000.0,23.81
9,東京都中野区中野５丁目２１番７,中野区,4370000.0,5410000.0,23.80



下落TOP10:


,location_text,city_name,price_2025,price_2026,pct_change
4063,神奈川県川崎市麻生区岡上６丁目１４６２番１６５,麻生区,142000.0,NaN,NaN
4064,神奈川県川崎市麻生区上麻生７丁目１７６番５,麻生区,NaN,173000.0,NaN
4065,神奈川県相模原市緑区町屋２丁目３３１９番１,緑区,122000.0,NaN,NaN
4066,神奈川県相模原市中央区上溝字乙四号３９０５番７外,中央区,NaN,115000.0,NaN
4067,神奈川県相模原市中央区青葉１丁目６２０２番４５,中央区,151000.0,NaN,NaN
4068,神奈川県相模原市中央区横山３丁目４９１２番４,中央区,165000.0,NaN,NaN
4069,神奈川県相模原市中央区千代田２丁目５０１７番１３,中央区,NaN,197000.0,NaN
4070,神奈川県相模原市南区東林間１丁目１８番２５,南区,NaN,288000.0,NaN
4071,神奈川県愛甲郡愛川町半原字下新久１８１５番３,愛川町,NaN,34200.0,NaN
4072,神奈川県愛甲郡愛川町中津字諏訪２２６６番２,愛川町,48500.0,NaN,NaN


## 5. 近傍地点比較（任意座標）

不動産投資シミュレーターへの接続イメージ: 物件座標を渡して近傍公示価格を参照する。

In [6]:
from analytics import find_nearby_points

if "df_all" in dir() and not df_all.empty:
    # 例: 東京駅付近 (lon=139.7671, lat=35.6812)
    TARGET_LON = 139.7671
    TARGET_LAT = 35.6812
    RADIUS_M = 1500
    YEAR = years[0] if years else None

    nearby = find_nearby_points(df_all, TARGET_LON, TARGET_LAT, radius_m=RADIUS_M, year=YEAR)
    print(f"東京駅付近 {RADIUS_M}m 以内の公示地点: {len(nearby)} 件")
    if not nearby.empty:
        display(nearby[["point_id", "location_text", "price_yen_per_sqm", "yoy_change_pct", "distance_m"]].head(10))

東京駅付近 1500m 以内の公示地点: 55 件


,point_id,location_text,price_yen_per_sqm,yoy_change_pct,distance_m
0,3008997,東京都千代田区丸の内２丁目２番１外,37600000.0,1.3,286.049300
1,3009079,東京都中央区八重洲１丁目１０５番５３,24300000.0,3.0,308.360318
2,3009041,東京都中央区京橋１丁目１番１外,24500000.0,3.4,372.571736
3,8008295,東京都中央区八重洲１丁目１０５番９,10500000.0,9.6,392.095926
4,17002710,東京都中央区日本橋３丁目２番３外,7230000.0,9.0,402.211019
5,10007919,東京都千代田区大手町２丁目４番２外,29800000.0,1.4,413.294001
6,3009089,東京都中央区京橋１丁目６番１外,18400000.0,2.8,538.983515
7,17002680,東京都中央区八重洲２丁目１０番１外,14300000.0,2.9,564.202210
8,3009084,東京都中央区日本橋２丁目１番１外,22700000.0,3.2,565.667401
9,14006769,東京都千代田区丸の内３丁目２番外,27500000.0,1.5,575.886104
